In [1]:
import os
import torch
import numpy as np
os.environ["KERAS_BACKEND"] = "torch"

In [2]:
if torch.backends.mps.is_available():
    print("Apple's MPS backend (used by PyTorch on M1/M2 Macs) does not support float64 (double precision).")
    mps_enabled = True
    torch.set_default_dtype(torch.float32)
    print("set default to float32")

Apple's MPS backend (used by PyTorch on M1/M2 Macs) does not support float64 (double precision).
set default to float32


In [3]:
import keras
from utils.project_utils import update_json, standard_plot, plot_confusion_matrix
from executors.executors import build_model_single_hidden, build_model_two_hidden
from utils.data_loader import get_monk_data

/Users/michaelbiggeri/Desktop/Informatica/Projects/Rage_Against_ML/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/michaelbiggeri/Desktop/Informatica/Projects/Rage_Against_ML/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Apple's MPS backend (used by PyTorch on M1/M2 Macs) does not support float64 (double precision).
set default to float32


## Part 1: Monk

In [4]:
# --- MONK ---
# 1. Load Data
monk_results = {}
batch_size = 64 # Explicitly define batch_size for Keras notebook

experiments = [
    {'id': 1, 'reg': False, 'name': 'Monk-1'},
    {'id': 2, 'reg': False, 'name': 'Monk-2'},
    {'id': 3, 'reg': False, 'name': 'Monk-3'},
    {'id': 3, 'reg': True, 'name': 'Monk-3-Reg'}
]

for exp in experiments:
    monk_id = exp['id']
    is_reg = exp['reg']
    task_name = exp['name']
    
    print(f"\n--- Analyzing {task_name} ---")
    train_loader, test_loader, input_size, output_size = get_monk_data(monk_id, batch_size)
    
    # Keras Model Builder
    def build_monk_model(hidden_units, learning_rate, momentum, l2_reg=0.0):
        reg = keras.regularizers.l2(l2_reg) if l2_reg > 0 else None
        model = keras.Sequential([
            keras.layers.Dense(hidden_units, activation='relu', input_shape=(input_size,), kernel_regularizer=reg),
            keras.layers.Dense(1, activation='sigmoid', kernel_regularizer=reg)
        ])
        optimizer = keras.optimizers.SGD(learning_rate=learning_rate, momentum=momentum)
        model.compile(optimizer=optimizer, loss='binary_crossentropy', metrics=['accuracy'])
        return model

    # Grid Search
    learning_rates = [0.01, 0.1]
    momentums = [0.9]
    hidden_units_list = [4, 8]
    l2_regs = [0.0] if not is_reg else [0.1, 0.5]
    epochs_list = [100]
    
    best_acc = 0.0
    best_config = {}
    best_history = {}
    
    for lr in learning_rates:
        for mom in momentums:
            for hu in hidden_units_list:
                for l2 in l2_regs:
                    for epochs in epochs_list:
                        # Clear session to avoid clutter
                        keras.backend.clear_session()
                        
                        model = build_monk_model(hu, lr, mom, l2)
                        
                        # Convert loaders to numpy for Keras
                        X_train = train_loader.dataset.tensors[0].numpy()
                        y_train = train_loader.dataset.tensors[1].numpy()
                        X_test = test_loader.dataset.tensors[0].numpy()
                        y_test = test_loader.dataset.tensors[1].numpy()
                        
                        history = model.fit(
                            X_train, y_train,
                            epochs=epochs,
                            batch_size=batch_size,
                            validation_data=(X_test, y_test),
                            verbose=0
                        )
                        
                        val_acc = history.history['val_accuracy'][-1]
                        if val_acc > best_acc:
                            best_acc = val_acc
                            best_config = {'lr': lr, 'momentum': mom, 'hidden_units': hu, 'l2_reg': l2, 'epochs': epochs}
                            best_history = history.history
    
    print(f"Best Config for {task_name}: {best_config}, Acc: {best_acc}")
    
    # Save Results
    standard_plot(best_history, f"{task_name} Best Model", f"{task_name.lower().replace('-', '_')}_keras.png")
    update_json(task_name, "Keras", {"test_accuracy": best_acc, "config": best_config})

    
    # Re-build best model to plot Confusion Matrix
    print(f"Generating Confusion Matrix for {task_name}...")
    keras.backend.clear_session()
    best_lr = best_config['lr']
    best_mom = best_config['momentum']
    best_hu = best_config['hidden_units']
    best_l2 = best_config['l2_reg']
    best_epochs = best_config['epochs']
    
    model = build_monk_model(best_hu, best_lr, best_mom, best_l2)
    
    X_train = train_loader.dataset.tensors[0].numpy()
    y_train = train_loader.dataset.tensors[1].numpy()
    X_test = test_loader.dataset.tensors[0].numpy()
    y_test = test_loader.dataset.tensors[1].numpy()
    
    model.fit(X_train, y_train, epochs=best_epochs, batch_size=batch_size, verbose=0)
    
    y_pred = model.predict(X_test, verbose=0)
    plot_confusion_matrix(y_test, y_pred, f"{task_name} Best Model", f"{task_name.lower().replace('-', '_')}_keras_cm.png")



--- Analyzing Monk-1 ---
Parsing MONK-1 data...


/Users/michaelbiggeri/Desktop/Informatica/Projects/Rage_Against_ML/.venv/lib/python3.9/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Best Config for Monk-1: {'lr': 0.1, 'momentum': 0.9, 'hidden_units': 4, 'l2_reg': 0.0, 'epochs': 100}, Acc: 1.0
Saved plot to monk_1_keras.png
Updated all_results.json for Task: Monk-1, Model: Keras
Generating Confusion Matrix for Monk-1...
Saved confusion matrix to monk_1_keras_cm.png

--- Analyzing Monk-2 ---
Parsing MONK-2 data...


/Users/michaelbiggeri/Desktop/Informatica/Projects/Rage_Against_ML/.venv/lib/python3.9/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Best Config for Monk-2: {'lr': 0.1, 'momentum': 0.9, 'hidden_units': 8, 'l2_reg': 0.0, 'epochs': 100}, Acc: 0.9814814925193787
Saved plot to monk_2_keras.png
Updated all_results.json for Task: Monk-2, Model: Keras
Generating Confusion Matrix for Monk-2...
Saved confusion matrix to monk_2_keras_cm.png

--- Analyzing Monk-3 ---
Parsing MONK-3 data...


/Users/michaelbiggeri/Desktop/Informatica/Projects/Rage_Against_ML/.venv/lib/python3.9/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Best Config for Monk-3: {'lr': 0.01, 'momentum': 0.9, 'hidden_units': 4, 'l2_reg': 0.0, 'epochs': 100}, Acc: 0.9722222089767456
Saved plot to monk_3_keras.png
Updated all_results.json for Task: Monk-3, Model: Keras
Generating Confusion Matrix for Monk-3...
Saved confusion matrix to monk_3_keras_cm.png

--- Analyzing Monk-3-Reg ---
Parsing MONK-3 data...


/Users/michaelbiggeri/Desktop/Informatica/Projects/Rage_Against_ML/.venv/lib/python3.9/site-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Best Config for Monk-3-Reg: {'lr': 0.01, 'momentum': 0.9, 'hidden_units': 8, 'l2_reg': 0.1, 'epochs': 100}, Acc: 0.9675925970077515
Saved plot to monk_3_reg_keras.png
Updated all_results.json for Task: Monk-3-Reg, Model: Keras
Generating Confusion Matrix for Monk-3-Reg...
Saved confusion matrix to monk_3_reg_keras_cm.png


# Part 2: ML-CUP

In [5]:
if keras.backend.backend() != "torch":
    print(f"warning: keras backend is set to {keras.backend.backend()}, restart jupyter kernel!!!!")
    raise RuntimeError()

In [6]:
keras.backend.backend()

'torch'

In [7]:
import json

global config
with open('./config/keras_nn.json') as keras_nn_config:
    config = json.load(keras_nn_config)
    print("config loaded")

config loaded


In [8]:
# Root-level fields
batch_size = config["batchSize"]
scaler_enabled = config["scaler"]["enabled"]
scaler_type = config["scaler"]["type"]
input_size = config["inputSize"]
output_size = config["outputSize"]
seed = config["seed"]

In [9]:
keras.utils.set_random_seed(seed)

In [10]:
%load_ext tensorboard
# now available at http://localhost:6006/?

In [15]:
# Dataset initialization
from utils.data_loader import get_ml_cup_data, split_dataloader, cv_fold_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, MaxAbsScaler

def _scaler():
    if not scaler_enabled:
        return None

    if scaler_type == "Standard":
        return StandardScaler()
    elif scaler_type == "MinMax":
        return MinMaxScaler()
    elif scaler_type == "Robust":
        return RobustScaler()
    elif scaler_type == "MaxAbsScaler":
        return MaxAbsScaler()
    else:
        return None

# We use validation_ratio=0.0 and scale_target=False to match the behavior 
# of the old data_loader_2 (which only did a Train/Test split and didn't scale targets).
# We unpack the result to get train_loader and internal_test_loader.
train_loader, _, test_loader, _, _, _= get_ml_cup_data(
    batch_size=batch_size,
    validation_ratio=0.0,
    test_ratio=0.20,  # Standard ratio from old loader
    scaler=_scaler(),
    scale_target=False # Old loader did not scale targets automatically
)

# Note: 'mps_enabled' logic is handled internally: data_loader.py automatically 
# uses float32 tensors (compatible with MPS) and checks for GPU memory pinning.

Parsing ML-CUP data (TR only)...
Data Split: Train=400, Val=0, Internal Test=100
Applying scaling StandardScaler to Inputs...


In [16]:
train_loader.dataset.X.shape, train_loader.dataset.y.shape

(torch.Size([400, 12]), torch.Size([400, 4]))

In [17]:
test_loader.dataset.X.shape, test_loader.dataset.y.shape

(torch.Size([100, 12]), torch.Size([100, 4]))

In [18]:
test_loader.dataset.y.shape[1]

4

In [19]:
import numpy as np

y_mean = train_loader.dataset.y.mean(axis=0)        # (4,)

y_pred_baseline = np.tile(y_mean, (len(train_loader.dataset.y), 1))

mee_errors = np.linalg.norm(train_loader.dataset.y - y_pred_baseline, axis=1)
mse_errors = np.square(train_loader.dataset.y - y_pred_baseline)

mee_baseline = mee_errors.mean()
mse_baseline = mse_errors.mean()

print("Baseline MEE:", mee_baseline)
print("Baseline MSE:", mse_baseline)

Baseline MEE: 35.881134
Baseline MSE: tensor(369.0105)


/var/folders/q_/svzf5dgd69q5h8h686ps7qxw0000gn/T/ipykernel_23520/559558886.py:7: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  mee_errors = np.linalg.norm(train_loader.dataset.y - y_pred_baseline, axis=1)
/var/folders/q_/svzf5dgd69q5h8h686ps7qxw0000gn/T/ipykernel_23520/559558886.py:8: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  mse_errors = np.square(train_loader.dataset.y - y_pred_baseline)


In [20]:
from losses import MeanEuclidianError

mee = MeanEuclidianError(name="mee", dtype=torch.float32)

In [21]:
train_dataset = train_loader.dataset
test_dataset = test_loader.dataset

## Randomized Search

In [ ]:
from scipy.stats import loguniform

param_distributions = {
    "reg__model__learning_rate": loguniform(1e-3, 1e-2),
    "reg__model__lambda_1": loguniform(3e-3, 1e-1),
    "reg__model__lambda_2": loguniform(3e-3, 1e-1),
    "reg__model__activation_1": ["relu", "gelu", "leaky_relu"],
    "reg__model__activation_2": ["relu", "gelu", "leaky_relu"],
    "reg__model__dropout_1": loguniform(0.2, 0.5),
    "reg__model__dropout_2": loguniform(0.2, 0.5),
    "reg__model__pca_input_size": [1, 2],
    "reg__pca__n_components": [1, 2],
    "reg__model__seed": [seed],
    "reg__model__output_size": [train_loader.dataset.y.shape[1]],
}

In [ ]:
from executors import RandomizedSearchRegressionExecutor

# using default param_distribution
rs_regression_executor = RandomizedSearchRegressionExecutor(
    train_loader=train_loader,
    units=[(2,2)],
    n_iter=1,
    epochs=1,
    use_PCA=True,
    loss="mean_squared_error",
    baseline=mse_baseline,
    scoring="neg_mean_squared_error",
    seed=seed,
    save_path="keras/models/rs/test",
    verbose=0,
    n_jobs=4
)

In [ ]:
rs_regression_executor.execute()

## Optuna

In [ ]:
import optuna
from executors import OptunaRegressorExecutor

optuna_executor = OptunaRegressorExecutor(
    train_loader=train_loader,
    units=[(12,12)],
    use_pca=True,
    pca_input_size=2,
    n_trials=100,
    epochs=1000,
    seed=seed,
    batch_size=80,
    n_splits=5,
    sampler=optuna.samplers.TPESampler(seed=seed, constant_liar=True, multivariate=True),
    optuna_base_path="keras/models/optuna/regression27-12",
    verbose=0,
    baseline=mse_baseline,
    n_jobs=4
)

In [ ]:
optuna_executor.execute()

In [ ]:
import utils.optuna as uoptuna

study = uoptuna.import_csv("keras/models/optuna/regression27-12/12x12/optuna_results.csv")

In [ ]:
from optuna.visualization import \
    plot_optimization_history, plot_param_importances, plot_parallel_coordinate, plot_contour

In [ ]:
plot_optimization_history(study)

In [ ]:
study.best_params

In [ ]:
plot_param_importances(study)

In [ ]:
plot_parallel_coordinate(study)

In [ ]:
plot_contour(study, params=['dropout_2', 'learning_rate'])

In [ ]:
from utils.plot import plot_optuna_vs_random

plot_optuna_vs_random(
    optuna_csv_path="keras/models/optuna/regression27-12/12x12/optuna_results.csv",
    rs_csv_path="keras/models/rs/regression27-12/12x12/cv_results_df.csv"
                      )

In [ ]:
# --- FINAL CUP EVALUATION ---
print("\n--- Final CUP Evaluation with Best Params ---")
# Assuming 'study' is available from previous cells
best_params = study.best_params
print("Best Params:", best_params)

# Extract params (assuming 12x12 architecture as per notebook default)
unit1 = 12
unit2 = 12

lr = best_params['learning_rate']
l1 = best_params['lambda_1']
act1 = best_params['activation_1']
drop1 = best_params['dropout_1']

l2 = best_params.get('lambda_2', 0.0)
act2 = best_params.get('activation_2', 'relu')
drop2 = best_params.get('dropout_2', 0.0)

meta = {"n_features_in_": input_size, "n_outputs_": output_size}

# Build Model
model = build_model_two_hidden(
    meta, unit1, unit2, seed, lr, drop1, drop2, l1, l2, act1, act2
)

# Train on Full Train Data (Train+Val)
# train_loader in notebook is already Train+Val (validation_ratio=0.0)
X_train = train_loader.dataset.X.numpy()
y_train = train_loader.dataset.y.numpy()
X_test = test_loader.dataset.X.numpy()
y_test = test_loader.dataset.y.numpy()

# Callbacks
callbacks = [
    keras.callbacks.EarlyStopping(monitor='loss', patience=50, restore_best_weights=True)
]

history = model.fit(
    X_train, y_train,
    epochs=1000, # Train longer for final
    batch_size=batch_size,
    verbose=0,
    callbacks=callbacks
)

# Evaluate
# MEE is in metrics (index 1 usually, index 0 is loss)
results = model.evaluate(X_test, y_test, verbose=0)
loss = results[0]
mee_score = results[1]
print(f"Final Internal Test MEE: {mee_score}")

# Save Results
update_json("CUP", "Keras", {"internal_test_MEE": mee_score})
standard_plot(history.history, "CUP Keras Training", "cup_keras.png")

# Save the Best Model
import os
os.makedirs("models", exist_ok=True)
model_save_path = "models/best_keras_mlcup.keras"
model.save(model_save_path)
print(f"Best Keras model saved to {model_save_path}")
